<a href="https://colab.research.google.com/github/zainababbasi5/MODEL-TRAINING/blob/master/train_all_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
"""
train_all_v3.py  --  upgraded pipeline (replaces train_all_v2.py)

What is new compared with v2
  1. ARIMA order chosen by an AIC grid search on TRAINING data only
     (d from an ADF unit-root test, then grid over p and q).
  2. XGBoost hyperparameters chosen by TimeSeriesSplit cross-validation
     on TRAINING windows only (gap = WINDOW so folds do not overlap).
  3. Library versions and GPU name are saved to results/versions.json.
  4. Epochs, best epoch, training time and parameter count are logged
     for every model and seed (results/training_log.csv).
  5. Optional second experiment with all OHLCV features
     (results_ohlcv/), plus a comparison with the Close-only run.

Colab:  !pip install xgboost statsmodels      (T4 GPU runtime recommended)
Run order: first RUN_OHLCV = False, check the output, then set True.
"""
import os, random, json, time, platform, warnings
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from scipy import stats
import sklearn
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import TimeSeriesSplit, ParameterGrid
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers
import xgboost
from xgboost import XGBRegressor
import statsmodels
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller

warnings.filterwarnings("ignore")

# ---------------- CONFIG ----------------
CSV_PATH = "/crude_oil_data (1).csv"
WINDOW = 60
TRAIN_FRAC, VAL_FRAC = 0.70, 0.15
SEEDS = [0, 1, 2, 3, 4]
EPOCHS, BATCH, LR = 100, 32, 1e-3
RUN_OHLCV = False                      # set True for the second experiment
OHLCV_COLS = ["Open", "High", "Low", "Close", "Volume"]
ARIMA_MAX_P, ARIMA_MAX_Q = 8, 2
XGB_GRID = dict(max_depth=[3, 4, 6], learning_rate=[0.03, 0.05, 0.1],
                n_estimators=[300, 500])


def set_seed(s):
    random.seed(s)
    np.random.seed(s)
    tf.random.set_seed(s)


# ---------------- VERSIONS (step 3) ----------------
versions = dict(
    python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__,
    scikit_learn=sklearn.__version__, tensorflow=tf.__version__,
    xgboost=xgboost.__version__, statsmodels=statsmodels.__version__,
    matplotlib=matplotlib.__version__,
    gpu=[d.name for d in tf.config.list_physical_devices("GPU")])
print("Versions:", versions)

# ---------------- DATA ----------------
df = pd.read_csv(CSV_PATH, skiprows=[1, 2])
df = df.rename(columns={df.columns[0]: "Date"})
df["Date"] = pd.to_datetime(df["Date"])
for c in OHLCV_COLS:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df = df.dropna(subset=["Close"]).sort_values("Date").reset_index(drop=True)
# other columns: forward-fill (past values only) so the row count stays identical
df[["Open", "High", "Low", "Volume"]] = df[["Open", "High", "Low", "Volume"]].ffill().fillna(0)

prices = df["Close"].values.astype("float64")
dates = df["Date"].values
n = len(prices)
i_tr, i_va = int(n * TRAIN_FRAC), int(n * (TRAIN_FRAC + VAL_FRAC))
print(f"Rows {n} | train {i_tr} | val {i_va - i_tr} | test {n - i_va}")
print(f"Range {df['Date'].iloc[0].date()} -> {df['Date'].iloc[-1].date()}")


# ---------------- HELPERS ----------------
def metrics(y, p):
    mse = mean_squared_error(y, p)
    return dict(MSE=mse, MAE=mean_absolute_error(y, p),
                RMSE=float(np.sqrt(mse)), R2=r2_score(y, p))


def dm_test(y, p1, p2):
    d = (y - p1) ** 2 - (y - p2) ** 2
    T = len(d)
    stat = d.mean() / np.sqrt(d.var(ddof=0) / T) * np.sqrt((T - 1) / T)
    return stat, 2 * (1 - stats.t.cdf(abs(stat), df=T - 1))


def build(kind, n_feat):
    m = models.Sequential([layers.Input(shape=(WINDOW, n_feat))])
    for units, ret in [(64, True), (32, False)]:
        if kind == "BiLSTM":
            m.add(layers.Bidirectional(layers.LSTM(units, return_sequences=ret)))
        elif kind == "LSTM":
            m.add(layers.LSTM(units, return_sequences=ret))
        else:
            m.add(layers.GRU(units, return_sequences=ret))
        m.add(layers.Dropout(0.2))
    m.add(layers.Dense(16, activation="relu"))
    m.add(layers.Dense(1))
    m.compile(optimizers.Adam(LR), loss="mse")
    return m


def select_arima(train_prices, out):
    """Step 1: d from ADF test, then AIC grid over p, q (training data only)."""
    d = 0
    if adfuller(train_prices)[1] > 0.05:
        d = 1
        if adfuller(np.diff(train_prices))[1] > 0.05:
            d = 2
    rows = []
    for p in range(ARIMA_MAX_P + 1):
        for q in range(ARIMA_MAX_Q + 1):
            try:
                r = ARIMA(train_prices, order=(p, d, q)).fit()
                rows.append(dict(p=p, d=d, q=q, aic=r.aic, bic=r.bic))
                print(f"  ARIMA({p},{d},{q}) AIC={r.aic:.1f}")
            except Exception as e:
                print(f"  ARIMA({p},{d},{q}) failed: {e}")
    tab = pd.DataFrame(rows).sort_values("aic").reset_index(drop=True)
    tab.to_csv(f"{out}/arima_aic_grid.csv", index=False)
    best = tab.iloc[0]
    return (int(best.p), int(best.d), int(best.q))


def tune_xgb(X2d, y, out):
    """Step 2: TimeSeriesSplit CV on training windows only."""
    tscv = TimeSeriesSplit(n_splits=5, gap=WINDOW)
    rows = []
    for g in ParameterGrid(XGB_GRID):
        errs = []
        for tr, va in tscv.split(X2d):
            m = XGBRegressor(subsample=0.8, colsample_bytree=0.8,
                             random_state=0, n_jobs=-1, **g)
            m.fit(X2d[tr], y[tr])
            errs.append(mean_squared_error(y[va], m.predict(X2d[va])))
        rows.append({**g, "cv_mse": float(np.mean(errs))})
        print("  XGB", g, "cv_mse=%.6f" % rows[-1]["cv_mse"])
    tab = pd.DataFrame(rows).sort_values("cv_mse").reset_index(drop=True)
    tab.to_csv(f"{out}/xgb_cv_grid.csv", index=False)
    best = {k: tab.iloc[0][k] for k in XGB_GRID}
    best = {k: (int(v) if k != "learning_rate" else float(v)) for k, v in best.items()}
    json.dump(best, open(f"{out}/xgb_best_params.json", "w"))
    return best


def make_figures(out, test_dates, y_true, preds, table, histories):
    DL = ["LSTM", "GRU", "BiLSTM"]
    fig, ax = plt.subplots(3, 1, figsize=(10, 9), sharex=True)
    for a, k in zip(ax, DL):
        a.plot(test_dates, y_true, label="Actual", lw=1.2)
        a.plot(test_dates, preds[k], label="Predicted", lw=1.0)
        a.set_title(f"{k}  (RMSE={table.loc[k, 'RMSE']:.4f}, R$^2$={table.loc[k, 'R2']:.4f})")
        a.set_ylabel("WTI price (USD)")
        a.legend()
    plt.tight_layout(); plt.savefig(f"{out}/fig2_actual_vs_pred.png", dpi=300); plt.close()

    fig, ax = plt.subplots(1, 3, figsize=(13, 4))
    for a, k in zip(ax, DL):
        a.scatter(y_true, preds[k], s=6, alpha=0.5)
        lo, hi = y_true.min(), y_true.max()
        a.plot([lo, hi], [lo, hi], "r--")
        a.set_title(f"{k}  (RMSE={table.loc[k, 'RMSE']:.4f}, R$^2$={table.loc[k, 'R2']:.4f})")
        a.set_xlabel("Actual"); a.set_ylabel("Predicted")
    plt.tight_layout(); plt.savefig(f"{out}/fig4_scatter.png", dpi=300); plt.close()

    fig, ax = plt.subplots(1, 3, figsize=(13, 4))
    for a, k in zip(ax, DL):
        a.plot(histories[k]["loss"], label="Train")
        a.plot(histories[k]["val_loss"], label="Val")
        a.set_title(f"{k} loss"); a.set_xlabel("Epoch"); a.set_ylabel("MSE (scaled)")
        a.legend()
    plt.tight_layout(); plt.savefig(f"{out}/fig3_loss_curves.png", dpi=300); plt.close()


# ---------------- ONE FULL EXPERIMENT ----------------
def run(feature_cols, tag, do_arima, figures):
    out = "results" if tag == "univariate" else f"results_{tag}"
    os.makedirs(out, exist_ok=True)
    json.dump(versions, open(f"{out}/versions.json", "w"), indent=2)
    print(f"\n===== EXPERIMENT: {tag}  features={feature_cols} -> {out}/ =====")

    F = df[feature_cols].values.astype("float64")
    ci = feature_cols.index("Close")
    nf = len(feature_cols)
    scaler = MinMaxScaler().fit(F[:i_tr])               # train only
    S = scaler.transform(F)
    lo_c, hi_c = scaler.data_min_[ci], scaler.data_max_[ci]
    inv = lambda a: np.asarray(a).ravel() * (hi_c - lo_c) + lo_c

    def windows(start, end):
        X, y = [], []
        for t in range(max(start, WINDOW), end):
            X.append(S[t - WINDOW:t, :])
            y.append(S[t, ci])
        return np.array(X), np.array(y)

    Xtr, ytr = windows(0, i_tr)
    Xva, yva = windows(i_tr, i_va)
    Xte, yte = windows(i_va, n)
    y_true = inv(yte)
    test_dates = dates[i_va:n]

    preds, histories, per_seed, log = {}, {}, {}, []
    preds["Naive"] = prices[i_va - 1:n - 1]

    # --- neural models (steps 3-4: log epochs, time, params)
    for kind in ["LSTM", "GRU", "BiLSTM"]:
        sp, sm = [], []
        for s in SEEDS:
            set_seed(s)
            m = build(kind, nf)
            t0 = time.time()
            h = m.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=EPOCHS,
                      batch_size=BATCH, verbose=0,
                      callbacks=[callbacks.EarlyStopping(patience=10, restore_best_weights=True)])
            secs = time.time() - t0
            p = inv(m.predict(Xte, verbose=0))
            sp.append(p); sm.append(metrics(y_true, p))
            log.append(dict(model=kind, seed=s, params=int(m.count_params()),
                            epochs_run=len(h.history["loss"]),
                            best_epoch=int(np.argmin(h.history["val_loss"])) + 1,
                            train_seconds=round(secs, 1), **sm[-1]))
            if s == SEEDS[0]:
                histories[kind] = h.history
            print(kind, "seed", s, {k: round(v, 4) for k, v in sm[-1].items()},
                  "epochs", log[-1]["epochs_run"], "sec", log[-1]["train_seconds"])
        per_seed[kind] = sm
        preds[kind] = np.mean(sp, axis=0)
    logdf = pd.DataFrame(log)
    logdf.to_csv(f"{out}/training_log.csv", index=False)
    logdf.groupby("model")[["params", "epochs_run", "best_epoch", "train_seconds"]].mean() \
        .to_csv(f"{out}/training_summary.csv")

    # --- XGBoost (step 2)
    X2tr, X2te = Xtr.reshape(len(Xtr), -1), Xte.reshape(len(Xte), -1)
    print("Tuning XGBoost with TimeSeriesSplit ...")
    best = tune_xgb(X2tr, ytr, out)
    print("Best XGBoost params:", best)
    xgb = XGBRegressor(subsample=0.8, colsample_bytree=0.8, random_state=0,
                       n_jobs=-1, **best)
    t0 = time.time()
    xgb.fit(X2tr, ytr)
    xgb_secs = time.time() - t0
    preds["XGBoost"] = inv(xgb.predict(X2te))

    # --- ARIMA (step 1), univariate experiment only
    if do_arima:
        print("ARIMA AIC grid on training data ...")
        order = select_arima(prices[:i_tr], out)
        print("Selected ARIMA order:", order)
        json.dump(dict(order=order), open(f"{out}/arima_order.json", "w"))
        fit = ARIMA(prices[:i_va], order=order).fit()    # train + validation
        ap = []
        for t in range(i_va, n):
            ap.append(float(fit.forecast(1)[0]))
            fit = fit.append([prices[t]], refit=False)
        preds["ARIMA"] = np.array(ap)

    for k, v in preds.items():
        assert len(v) == len(y_true), f"{k}: {len(v)} vs {len(y_true)}"

    # --- table
    order_models = [m for m in ["Naive", "ARIMA", "XGBoost", "LSTM", "GRU", "BiLSTM"] if m in preds]
    rows = []
    for name in order_models:
        r = metrics(y_true, preds[name]); r["Model"] = name
        if name in per_seed:
            for k in ["RMSE", "R2"]:
                vals = [d[k] for d in per_seed[name]]
                r[k + "_seed_mean"] = np.mean(vals); r[k + "_seed_std"] = np.std(vals)
        rows.append(r)
    table = pd.DataFrame(rows).set_index("Model")
    table.to_csv(f"{out}/table2_metrics.csv")
    print(table.round(4))

    # --- DM tests
    dm = []
    for i, a in enumerate(order_models):
        for b in order_models[i + 1:]:
            s_, p_ = dm_test(y_true, preds[a], preds[b])
            dm.append(dict(A=a, B=b, DM=s_, p=p_))
    pd.DataFrame(dm).to_csv(f"{out}/dm_tests.csv", index=False)
    print(pd.DataFrame(dm).round(4))

    pd.DataFrame({"date": test_dates, "actual": y_true, **preds}).to_csv(
        f"{out}/predictions.csv", index=False)
    json.dump(dict(tag=tag, features=feature_cols, n=int(n), train=int(i_tr),
                   val=int(i_va - i_tr), test=int(n - i_va), window=WINDOW,
                   seeds=SEEDS, xgb_fit_seconds=round(xgb_secs, 1)),
              open(f"{out}/run_config.json", "w"), indent=2)
    if figures:
        make_figures(out, test_dates, y_true, preds, table, histories)
    return dict(preds=preds, table=table, y_true=y_true)


# ---------------- RUN ----------------
res_u = run(["Close"], "univariate", do_arima=True, figures=True)

if RUN_OHLCV:
    res_o = run(OHLCV_COLS, "ohlcv", do_arima=False, figures=False)
    assert np.allclose(res_u["y_true"], res_o["y_true"], atol=1e-6)
    rows = []
    for k in ["XGBoost", "LSTM", "GRU", "BiLSTM"]:
        s_, p_ = dm_test(res_u["y_true"], res_o["preds"][k], res_u["preds"][k])
        rows.append(dict(model=k,
                         rmse_close_only=res_u["table"].loc[k, "RMSE"],
                         rmse_ohlcv=res_o["table"].loc[k, "RMSE"],
                         rmse_naive=res_u["table"].loc["Naive", "RMSE"],
                         DM_ohlcv_vs_close_only=s_, p=p_))   # negative: OHLCV better
    cmp_ = pd.DataFrame(rows)
    cmp_.to_csv("results_ohlcv/compare_with_close_only.csv", index=False)
    print(cmp_.round(4))

print("Done.")

Versions: {'python': '3.13.16', 'numpy': '2.4.6', 'pandas': '2.2.3', 'scikit_learn': '1.6.1', 'tensorflow': '2.21.0', 'xgboost': '3.4.1', 'statsmodels': '0.15.0', 'matplotlib': '3.10.0', 'gpu': ['/physical_device:GPU:0']}
Rows 6435 | train 4504 | val 965 | test 966
Range 2000-08-23 -> 2026-04-10

===== EXPERIMENT: univariate  features=['Close'] -> results/ =====
LSTM seed 0 {'MSE': 13.6373, 'MAE': 2.8062, 'RMSE': 3.6929, 'R2': 0.893} epochs 32 sec 57.9


KeyboardInterrupt: 

usage: colab_kernel_launcher.py [-h]
                                [--exp {all,paramatch,hpsearch,returns,multistep,exog}]
                                [--csv CSV] [--exog EXOG]
                                [--exog_csv EXOG_CSV]
                                [--seeds SEEDS [SEEDS ...]]
                                [--search_seeds SEARCH_SEEDS [SEARCH_SEEDS ...]]
                                [--n_configs N_CONFIGS]
                                [--horizons HORIZONS [HORIZONS ...]]
                                [--ms_models MS_MODELS [MS_MODELS ...]]
                                [--epochs EPOCHS] [--synthetic]
colab_kernel_launcher.py: error: unrecognized arguments: -f /root/.local/share/jupyter/runtime/kernel-087820d0-44a5-4fc0-861d-f1a14de2e336.json


SystemExit: 2

In [ ]:
   !ls -la /content/results

total 1480
drwxr-xr-x 2 root root   4096 Oct 25 12:25 .
drwxr-xr-x 1 root root   4096 Oct 25 12:06 ..
-rw-r--r-- 1 root root   1196 Oct 25 12:24 arima_aic_grid.csv
-rw-r--r-- 1 root root     20 Oct 25 12:24 arima_order.json
-rw-r--r-- 1 root root    695 Oct 25 12:25 dm_tests.csv
-rw-r--r-- 1 root root 689585 Oct 25 12:25 fig2_actual_vs_pred.png
-rw-r--r-- 1 root root 236024 Oct 25 12:25 fig3_loss_curves.png
-rw-r--r-- 1 root root 399466 Oct 25 12:25 fig4_scatter.png
-rw-r--r-- 1 root root 131145 Oct 25 12:25 predictions.csv
-rw-r--r-- 1 root root    211 Oct 25 12:25 run_config.json
-rw-r--r-- 1 root root    800 Oct 25 12:25 table2_metrics.csv
-rw-r--r-- 1 root root   1547 Oct 25 12:21 training_log.csv
-rw-r--r-- 1 root root    137 Oct 25 12:21 training_summary.csv
-rw-r--r-- 1 root root    239 Oct 25 12:06 versions.json
-rw-r--r-- 1 root root     60 Oct 25 12:23 xgb_best_params.json
-rw-r--r-- 1 root root    611 Oct 25 12:23 xgb_cv_grid.csv


In [ ]:
"""
Run AFTER train_all_v2.py (needs the ./results folder).
No retraining: reads results/predictions.csv and results/table2_metrics.csv.

Adds: naive baseline, updated table, DM tests incl. naive, and 5 extra figures:
  fig1_data_split.png      full price series with train/val/test regions
  fig5_rmse_comparison.png RMSE bar chart (+ seed mean +- std for deep models)
  fig6_all_models_zoom.png all models vs actual on the last 200 test days
  fig7_residuals.png       residual boxplots + histograms
  fig8_dm_heatmap.png      Diebold-Mariano heatmap
  fig9_pipeline.png        methodology pipeline diagram
"""
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from scipy import stats
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

CSV_PATH  = "/crude_oil_data (1).csv"
DATE_COL, PRICE_COL = "Date", "Close"
TRAIN_FRAC, VAL_FRAC = 0.70, 0.15
R = "results"

# ---------- load raw data exactly like train_all_v2.py ----------
df = pd.read_csv(CSV_PATH, skiprows=[1, 2])
df = df.rename(columns={df.columns[0]: DATE_COL})
df[DATE_COL] = pd.to_datetime(df[DATE_COL])
df[PRICE_COL] = pd.to_numeric(df[PRICE_COL], errors="coerce")
df = df[[DATE_COL, PRICE_COL]].dropna().sort_values(DATE_COL).reset_index(drop=True)
prices = df[PRICE_COL].values.astype("float64")
dates = df[DATE_COL].values
n = len(prices)
i_tr, i_va = int(n * TRAIN_FRAC), int(n * (TRAIN_FRAC + VAL_FRAC))

# ---------- saved predictions ----------
P = pd.read_csv(f"{R}/predictions.csv", parse_dates=["date"])
y = P["actual"].values
assert len(P) == n - i_va, "predictions.csv does not match the CSV"
assert np.allclose(prices[i_va:n], y, atol=1e-3), "actual column mismatch"

# ---------- naive baseline: tomorrow = today ----------
P["Naive"] = prices[i_va - 1:n - 1]
models = ["Naive", "ARIMA", "XGBoost", "LSTM", "GRU", "BiLSTM"]
DL = ["LSTM", "GRU", "BiLSTM"]


def metrics(y_, p_):
    mse = mean_squared_error(y_, p_)
    return dict(MSE=mse, MAE=mean_absolute_error(y_, p_),
                RMSE=float(np.sqrt(mse)), R2=r2_score(y_, p_))


T = pd.DataFrame({m: metrics(y, P[m].values) for m in models}).T
T.index.name = "Model"
old = pd.read_csv(f"{R}/table2_metrics.csv", index_col="Model")
for c in ["RMSE_seed_mean", "RMSE_seed_std", "R2_seed_mean", "R2_seed_std"]:
    if c in old.columns:
        T[c] = old[c]
T.to_csv(f"{R}/table2_with_naive.csv")
print(T.round(4))


# ---------- Diebold-Mariano (squared error, h=1, Harvey correction) ----------
def dm_test(y_, p1, p2):
    d = (y_ - p1) ** 2 - (y_ - p2) ** 2
    L = len(d)
    stat = d.mean() / np.sqrt(d.var(ddof=0) / L) * np.sqrt((L - 1) / L)
    return stat, 2 * (1 - stats.t.cdf(abs(stat), df=L - 1))


k = len(models)
S, PV = np.zeros((k, k)), np.ones((k, k))
rows = []
for i in range(k):
    for j in range(k):
        if i != j:
            S[i, j], PV[i, j] = dm_test(y, P[models[i]].values, P[models[j]].values)
            if i < j:
                rows.append(dict(A=models[i], B=models[j], DM=S[i, j], p=PV[i, j]))
pd.DataFrame(rows).to_csv(f"{R}/dm_tests_with_naive.csv", index=False)
print(pd.DataFrame(rows).round(4))

# ---------- Fig 1: data split ----------
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(dates, prices, lw=0.8, color="black")
ax.axvspan(dates[0], dates[i_tr - 1], alpha=0.15, color="tab:blue", label="Train (70%)")
ax.axvspan(dates[i_tr], dates[i_va - 1], alpha=0.15, color="tab:orange", label="Validation (15%)")
ax.axvspan(dates[i_va], dates[-1], alpha=0.15, color="tab:green", label="Test (15%)")
ax.set_ylabel("WTI close (USD)")
ax.set_title("WTI crude oil price and chronological data split")
ax.legend(loc="upper left")
plt.tight_layout(); plt.savefig(f"{R}/fig1_data_split.png", dpi=300); plt.close()

# ---------- Fig 5: RMSE comparison ----------
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
order = T["RMSE"].sort_values().index
bars = ax[0].bar(order, T.loc[order, "RMSE"], color="tab:blue")
ax[0].bar_label(bars, fmt="%.3f", padding=2)
ax[0].set_ylabel("RMSE (USD)")
ax[0].set_title("(a) Test RMSE (deep models: seed-averaged prediction)")
ax[0].tick_params(axis="x", rotation=20)
if "RMSE_seed_mean" in T.columns:
    ax[1].bar(DL, T.loc[DL, "RMSE_seed_mean"], yerr=T.loc[DL, "RMSE_seed_std"],
              capsize=6, color="tab:orange")
    for xi, m in enumerate(DL):
        ax[1].text(xi, T.loc[m, "RMSE_seed_mean"] + T.loc[m, "RMSE_seed_std"] + 0.05,
                   f"{T.loc[m, 'RMSE_seed_mean']:.2f}±{T.loc[m, 'RMSE_seed_std']:.2f}",
                   ha="center")
    ax[1].set_ylabel("RMSE (USD)")
    ax[1].set_title("(b) Per-seed RMSE: mean ± std")
plt.tight_layout(); plt.savefig(f"{R}/fig5_rmse_comparison.png", dpi=300); plt.close()

# ---------- Fig 6: all models, last 200 days ----------
z = slice(-200, None)
fig, ax = plt.subplots(figsize=(11, 4.8))
ax.plot(P["date"][z], y[z], color="black", lw=1.8, label="Actual")
for m in ["ARIMA", "XGBoost", "LSTM", "GRU", "BiLSTM"]:
    ax.plot(P["date"][z], P[m].values[z], lw=1.0, label=m)
ax.set_ylabel("WTI close (USD)")
ax.set_title("All models vs actual (last 200 test days)")
ax.legend(ncol=3)
plt.tight_layout(); plt.savefig(f"{R}/fig6_all_models_zoom.png", dpi=300); plt.close()

# ---------- Fig 7: residuals ----------
res = {m: y - P[m].values for m in models}
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].boxplot([res[m] for m in models], labels=models, showfliers=True)
ax[0].axhline(0, color="red", ls="--", lw=0.8)
ax[0].set_ylabel("Residual = actual - predicted (USD)")
ax[0].set_title("(a) Residual distribution")
ax[0].tick_params(axis="x", rotation=20)
for m in ["ARIMA", "XGBoost", "BiLSTM"]:
    ax[1].hist(res[m], bins=50, alpha=0.45, label=m)
ax[1].set_xlabel("Residual (USD)"); ax[1].set_ylabel("Count")
ax[1].set_title("(b) Residual histograms"); ax[1].legend()
plt.tight_layout(); plt.savefig(f"{R}/fig7_residuals.png", dpi=300); plt.close()

# ---------- Fig 8: DM heatmap ----------
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(S, cmap="RdBu", vmin=-12, vmax=12)
ax.set_xticks(range(k)); ax.set_xticklabels(models, rotation=30)
ax.set_yticks(range(k)); ax.set_yticklabels(models)
for i in range(k):
    for j in range(k):
        if i == j:
            ax.text(j, i, "-", ha="center", va="center")
        else:
            star = "***" if PV[i, j] < 0.001 else "**" if PV[i, j] < 0.01 else "*" if PV[i, j] < 0.05 else "ns"
            ax.text(j, i, f"{S[i, j]:.1f}\n{star}", ha="center", va="center", fontsize=8)
ax.set_title("Diebold-Mariano statistic\n(negative: row model more accurate than column model)")
plt.colorbar(im, fraction=0.046)
plt.tight_layout(); plt.savefig(f"{R}/fig8_dm_heatmap.png", dpi=300); plt.close()

# ---------- Fig 9: pipeline diagram ----------
steps = ["Yahoo Finance\nWTI daily close\n(2000-2026)", "Chronological\nsplit\n70/15/15",
         "Min-Max scaling\n(fit on train only)", "Sliding window\n(60 days -> next day)",
         "Models:\nARIMA, XGBoost,\nLSTM, GRU, BiLSTM", "Training\nAdam + early stopping\n(multiple seeds)",
         "Evaluation:\nRMSE, MAE, R², DM test"]
fig, ax = plt.subplots(figsize=(13, 2.6))
ax.set_xlim(0, len(steps)); ax.set_ylim(0, 1); ax.axis("off")
for i, s in enumerate(steps):
    ax.add_patch(FancyBboxPatch((i + 0.06, 0.2), 0.82, 0.6, boxstyle="round,pad=0.02",
                                fc="#dbe9f6", ec="#2f5f8a"))
    ax.text(i + 0.47, 0.5, s, ha="center", va="center", fontsize=8)
    if i < len(steps) - 1:
        ax.annotate("", xy=(i + 1.05, 0.5), xytext=(i + 0.91, 0.5),
                    arrowprops=dict(arrowstyle="->", color="#2f5f8a"))
plt.tight_layout(); plt.savefig(f"{R}/fig9_pipeline.png", dpi=300); plt.close()

print("Done. New files are in ./results")

            MSE     MAE    RMSE      R2  RMSE_seed_mean  RMSE_seed_std  \
Model                                                                    
Naive    3.8377  1.3584  1.9590  0.9699             NaN            NaN   
ARIMA    3.9227  1.3712  1.9806  0.9692             NaN            NaN   
XGBoost  4.2674  1.4317  2.0658  0.9665             NaN            NaN   
LSTM     9.8046  2.3114  3.1312  0.9231          3.3848         0.4563   
GRU      8.2658  2.2162  2.8750  0.9352          3.1575         0.7713   
BiLSTM   7.1139  1.9605  2.6672  0.9442          2.7979         0.2127   

         R2_seed_mean  R2_seed_std  
Model                               
Naive             NaN          NaN  
ARIMA             NaN          NaN  
XGBoost           NaN          NaN  
LSTM           0.9085       0.0228  
GRU            0.9171       0.0387  
BiLSTM         0.9382       0.0096  
          A        B       DM       p
0     Naive    ARIMA  -1.3110  0.1902
1     Naive  XGBoost  -2.9541  0.00

In [ ]:
!zip -r results.zip results
from google.colab import files
files.download("results.zip")

  adding: results/ (stored 0%)
  adding: results/fig9_pipeline.png (deflated 13%)
  adding: results/fig7_residuals.png (deflated 19%)
  adding: results/table2_with_naive.csv (deflated 44%)
  adding: results/dm_tests_with_naive.csv (deflated 49%)
  adding: results/xgb_best_params.json (deflated 7%)
  adding: results/training_summary.csv (deflated 16%)
  adding: results/arima_aic_grid.csv (deflated 55%)
  adding: results/fig5_rmse_comparison.png (deflated 21%)
  adding: results/arima_order.json (stored 0%)
  adding: results/dm_tests.csv (deflated 49%)
  adding: results/table2_metrics.csv (deflated 44%)
  adding: results/run_config.json (deflated 36%)
  adding: results/fig4_scatter.png (deflated 9%)
  adding: results/fig6_all_models_zoom.png (deflated 8%)
  adding: results/xgb_cv_grid.csv (deflated 55%)
  adding: results/training_log.csv (deflated 48%)
  adding: results/fig1_data_split.png (deflated 8%)
  adding: results/predictions.csv (deflated 60%)
  adding: results/fig8_dm_heatmap.png

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
"""
Extra experiments for the WTI next-day forecasting paper (reviewer-proofing).

  A  parammatch : parameter-matched LSTM vs BiLSTM (removes the capacity confound)
  B  hpsearch   : validation-only hyperparameter search for LSTM / GRU / BiLSTM
  C  returns    : forecast next-day log RETURNS instead of price levels
  D  multistep  : direct multi-step price forecasts (h = 5, 10, 20 days)
  E  exog       : add ONE exogenous variable (default: US dollar index, DX-Y.NYB)

Same protocol as the paper: chronological 70/15/15 split, scaling fitted on the
training block only, causal windows, early stopping on validation, 5 seeds,
seed-averaged forecasts, DM test (HLN correction) + Holm correction.

Google Colab (GPU runtime):
    !pip -q install yfinance
    !python extra_experiments.py --exp paramatch      # ~15 min
    !python extra_experiments.py --exp hpsearch       # ~1.5-3 h (use --n_configs to shrink)
    !python extra_experiments.py --exp returns        # ~20 min
    !python extra_experiments.py --exp multistep      # ~1 h
    !python extra_experiments.py --exp exog           # ~20 min
    !python extra_experiments.py --exp all

Use your own CSV (columns Date, Close) for the exact same data as the paper:
    !python extra_experiments.py --exp all --csv wti.csv
Results go to ./results/ as CSV + ready-to-paste LaTeX tables.
"""
import argparse, json, os, random, time
import numpy as np
import pandas as pd
from scipy import stats

import tensorflow as tf
from tensorflow.keras import layers, Model, Input, callbacks, optimizers
import xgboost as xgb

# ---- EDIT THIS when pasting the script into a Colab/Jupyter cell -------------
# (ignored when run from the command line with !python extra_experiments.py ...)
# Experiments run one after another; hpsearch is last because it is the longest.
NOTEBOOK_ARGS = ["--exp", "paramatch", "returns", "hpsearch",
                 "--csv", "/crude_oil_data (1).csv"]
# Shorter hyperparameter search if Colab time is tight: add "--n_configs", "8"
# ------------------------------------------------------------------------------

OUT = "results"
SEEDS = [0, 1, 2, 3, 4]
DEFAULT = dict(u1=64, u2=32, drop=0.2, lr=1e-3, batch=32, T=60)   # paper's settings
XGB_PARAMS = dict(n_estimators=300, max_depth=3, learning_rate=0.03,
                  subsample=0.8, colsample_bytree=0.8, random_state=0)


# ----------------------------------------------------------------------------- data
def _close(df):
    c = df["Close"]
    if isinstance(c, pd.DataFrame):          # new yfinance returns MultiIndex columns
        c = c.iloc[:, 0]
    return c.dropna().sort_index()


def read_price_csv(path):
    """Read a price CSV in any common layout: plain (Date, Close), lower-case names,
    'Price'/'Adj Close' columns, or the 3-header-row CSV that yfinance writes."""
    import glob
    if not os.path.exists(path):
        base = os.path.basename(path).split(" (")[0].split(".")[0]
        hits = sorted(glob.glob(f"/content/**/{base}*.csv", recursive=True)) or \
               sorted(glob.glob(f"**/{base}*.csv", recursive=True))
        if not hits:
            raise FileNotFoundError(
                f"{path} not found. Upload it via the Colab file panel and put its exact path in "
                f"NOTEBOOK_ARGS. CSV files found: {glob.glob('/content/**/*.csv', recursive=True)}")
        print(f"'{path}' not found, using '{hits[0]}'")
        path = hits[0]
    df = pd.read_csv(path)
    # yfinance multi-header CSV: rows like 'Ticker,CL=F,...' and 'Date,,,' under the header
    first = str(df.iloc[0, 0]).strip().lower()
    if first in ("ticker", "date", "") or first == "nan":
        df = pd.read_csv(path, skiprows=[1, 2] if str(df.iloc[1, 0]).strip().lower() == "date"
                         else [1])
    cols = {c.lower().strip(): c for c in df.columns}
    date_col = next((cols[k] for k in ("date", "datetime", "time", "timestamp", "price")
                     if k in cols and pd.to_datetime(df[cols[k]], errors="coerce").notna().mean() > 0.9),
                    df.columns[0])
    close_col = next((cols[k] for k in ("close", "close/last", "adj close", "adj_close", "price", "value")
                      if k in cols and cols[k] != date_col), None)
    if close_col is None:
        raise KeyError(f"No closing-price column found. Columns are: {list(df.columns)}")
    out = pd.Series(pd.to_numeric(df[close_col].astype(str).str.replace(r"[,$\s]", "", regex=True),
                                  errors="coerce").values,
                    index=pd.to_datetime(df[date_col], errors="coerce"))
    out = out[out.index.notna()].dropna().sort_index()
    out = out[~out.index.duplicated(keep="last")]
    print(f"Loaded {path}: date column '{date_col}', price column '{close_col}', {len(out)} rows")
    return out


def load_series(ticker, csv=None, start="2000-08-23", end="2026-04-11"):
    if csv:
        return read_price_csv(csv)
    import yfinance as yf
    return _close(yf.download(ticker, start=start, end=end, auto_adjust=False, progress=False))


def synthetic(n=6435, seed=0):
    """Random-walk test series (only for checking that the script runs)."""
    rng = np.random.default_rng(seed)
    p = 60 + np.cumsum(rng.normal(0, 1.5, n))
    idx = pd.bdate_range("2000-08-23", periods=n)
    return pd.Series(np.abs(p) + 20, index=idx)


def split_sizes(n):
    n_tr = int(0.70 * n)
    n_va = int(0.15 * n)
    return n_tr, n_va, n - n_tr - n_va


class MinMax:
    def fit(self, x):
        self.lo, self.hi = np.min(x, 0), np.max(x, 0)
        return self

    def tf(self, x):
        return (x - self.lo) / (self.hi - self.lo)

    def inv(self, x):
        return x * (self.hi - self.lo) + self.lo


# ----------------------------------------------------------------------------- tasks
# A "task" defines inputs (feat), the scaled target y[j], the horizon h, and how to turn
# a scaled prediction for target day j back into a USD price forecast.
# For target day j the input window is feat[j-h+1-T : j-h+1]  (last observed day = j-h).

def price_task(p, h=1, exog=None):
    n = len(p)
    n_tr, _, _ = split_sizes(n)
    cols = [p] if exog is None else [p, exog]
    raw = np.column_stack(cols).astype("float64")
    sc = MinMax().fit(raw[:n_tr])                      # training block only
    feat = sc.tf(raw)
    psc = MinMax().fit(p[:n_tr, None])
    y = psc.tf(p[:, None])[:, 0]
    return dict(feat=feat, y=y, h=h, p=p, start=1,
                to_price=lambda pred, j: psc.inv(pred[:, None])[:, 0])


def return_task(p):
    n = len(p)
    n_tr, _, _ = split_sizes(n)
    r = np.zeros(n)
    r[1:] = np.diff(np.log(p))
    mu, sd = r[1:n_tr].mean(), r[1:n_tr].std()       # training block only
    z = (r - mu) / sd
    return dict(feat=z[:, None], y=z, h=1, p=p, start=1,
                to_price=lambda pred, j: p[j - 1] * np.exp(pred * sd + mu))


def target_index(task, T):
    n = len(task["p"])
    n_tr, n_va, _ = split_sizes(n)
    first = T + task["h"] - 1 + task["start"]
    return (np.arange(first, n_tr),
            np.arange(n_tr, n_tr + n_va),
            np.arange(n_tr + n_va, n))


def windows(task, T, idx):
    f, h = task["feat"], task["h"]
    X = np.stack([f[j - h + 1 - T: j - h + 1] for j in idx]).astype("float32")
    return X, task["y"][idx].astype("float32")


# ----------------------------------------------------------------------------- models
def lstm_p(u, n):  return 4 * (u * (u + n) + u)
def gru_p(u, n):   return 3 * (u * (u + n) + 2 * u)          # keras reset_after=True


def count_params(kind, u1, u2, nfeat=1):
    f = gru_p if kind == "gru" else lstm_p
    k = 2 if kind == "bilstm" else 1
    l1 = k * f(u1, nfeat)
    l2 = k * f(u2, k * u1)
    return l1 + l2 + (k * u2 * 16 + 16) + 17


def match_units(kind, target, nfeat=1):
    """Find (u1, u2) with u2 = u1/2 (same shape as the paper) closest to `target` params."""
    best = None
    for u1 in range(8, 257, 2):
        u2 = u1 // 2
        c = count_params(kind, u1, u2, nfeat)
        if best is None or abs(c - target) < abs(best[2] - target):
            best = (u1, u2, c)
    return best


def build(kind, nfeat, T, u1, u2, drop, lr, **_):
    L = layers.GRU if kind == "gru" else layers.LSTM

    def rec(u, seq):
        l = L(u, return_sequences=seq)
        return layers.Bidirectional(l) if kind == "bilstm" else l

    inp = Input((T, nfeat))
    x = rec(u1, True)(inp)
    x = layers.Dropout(drop)(x)
    x = rec(u2, False)(x)
    x = layers.Dropout(drop)(x)
    x = layers.Dense(16, activation="relu")(x)
    out = layers.Dense(1)(x)
    m = Model(inp, out)
    m.compile(optimizers.Adam(lr), loss="mse")
    return m


def run_nn(task, kind, cfg, seeds=SEEDS, epochs=100, verbose=True):
    """Train one config over several seeds. Returns val/test price forecasts (seed-averaged)
    plus per-seed test RMSE."""
    T = cfg["T"]
    itr, iva, ite = target_index(task, T)
    Xtr, ytr = windows(task, T, itr)
    Xva, yva = windows(task, T, iva)
    Xte, _ = windows(task, T, ite)
    p = task["p"]
    va_preds, te_preds, per_seed, n_params = [], [], [], None
    for s in seeds:
        tf.keras.backend.clear_session()
        tf.keras.utils.set_random_seed(s)
        m = build(kind, Xtr.shape[2], **cfg)
        n_params = m.count_params()
        es = callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)
        t0 = time.time()
        hist = m.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=epochs,
                     batch_size=cfg["batch"], shuffle=True, callbacks=[es], verbose=0)
        pv = task["to_price"](m.predict(Xva, verbose=0)[:, 0], iva)
        pt = task["to_price"](m.predict(Xte, verbose=0)[:, 0], ite)
        va_preds.append(pv); te_preds.append(pt)
        per_seed.append(rmse(p[ite], pt))
        if verbose:
            print(f"    {kind:7s} seed {s}: epochs={len(hist.history['loss']):3d} "
                  f"test RMSE={per_seed[-1]:.4f} ({time.time()-t0:.0f}s)", flush=True)
    return dict(val=np.mean(va_preds, 0), test=np.mean(te_preds, 0),
                per_seed=np.array(per_seed), params=n_params, ite=ite, iva=iva)


def run_xgb(task, T=60):
    itr, iva, ite = target_index(task, T)
    Xtr, ytr = windows(task, T, itr)
    Xte, _ = windows(task, T, ite)
    m = xgb.XGBRegressor(**XGB_PARAMS)
    m.fit(Xtr.reshape(len(Xtr), -1), ytr)
    return task["to_price"](m.predict(Xte.reshape(len(Xte), -1)), ite), ite


# ----------------------------------------------------------------------------- metrics
def rmse(y, f): return float(np.sqrt(np.mean((y - f) ** 2)))
def mae(y, f):  return float(np.mean(np.abs(y - f)))


def r2(y, f):   return float(1 - np.sum((y - f) ** 2) / np.sum((y - y.mean()) ** 2))


def dm_test(e1, e2, h=1, nw_lags=None):
    """Diebold-Mariano with Harvey-Leybourne-Newbold correction.
    h-step forecasts: rectangular long-run variance with h-1 lags (standard DM).
    nw_lags: if given, Newey-West (Bartlett) variance with that many lags instead.
    Negative stat => model 1 has the smaller squared error."""
    d = e1 ** 2 - e2 ** 2
    n = len(d)
    dc = d - d.mean()
    L = h - 1 if nw_lags is None else nw_lags
    lrv = dc @ dc / n
    for k in range(1, L + 1):
        w = 1.0 if nw_lags is None else 1 - k / (L + 1)
        lrv += 2 * w * (dc[k:] @ dc[:-k]) / n
    if lrv <= 0:
        lrv = dc @ dc / n
    hln = np.sqrt((n + 1 - 2 * h + h * (h - 1) / n) / n)
    stat = d.mean() / np.sqrt(lrv / n) * hln
    return float(stat), float(2 * stats.t.sf(abs(stat), df=n - 1))


def holm(pvals):
    p = np.asarray(pvals)
    order = np.argsort(p)
    adj = np.empty_like(p)
    running = 0
    for rank, i in enumerate(order):
        running = max(running, (len(p) - rank) * p[i])
        adj[i] = min(1.0, running)
    return adj


def stars(p):
    return "***" if p < 1e-3 else "**" if p < 1e-2 else "*" if p < 0.05 else "ns"


def metrics_table(y, forecasts, per_seed=None, params=None):
    rows = []
    for name, f in forecasts.items():
        r = dict(Model=name, MSE=np.mean((y - f) ** 2), MAE=mae(y, f), RMSE=rmse(y, f), R2=r2(y, f))
        if per_seed and name in per_seed:
            r["PerSeedRMSE"] = f"{per_seed[name].mean():.3f} ± {per_seed[name].std():.3f}"
        if params and name in params:
            r["Params"] = params[name]
        rows.append(r)
    return pd.DataFrame(rows)


def dm_table(y, forecasts, h=1, pairs=None):
    names = list(forecasts)
    pairs = pairs or [(a, b) for i, a in enumerate(names) for b in names[i + 1:]]
    rows = []
    for a, b in pairs:
        s, p = dm_test(y - forecasts[a], y - forecasts[b], h=h)
        s6, p6 = dm_test(y - forecasts[a], y - forecasts[b], h=h, nw_lags=max(6, h))
        rows.append(dict(A=a, B=b, DM=s, p=p, DM_NW=s6, p_NW=p6))
    df = pd.DataFrame(rows)
    df["p_Holm"] = holm(df["p"].values)
    df["sig"] = [stars(p) for p in df["p_Holm"]]
    return df


def save(df, name, caption):
    os.makedirs(OUT, exist_ok=True)
    df.to_csv(f"{OUT}/{name}.csv", index=False)
    with open(f"{OUT}/{name}.tex", "w") as fh:
        fh.write(df.to_latex(index=False, float_format="%.4f", caption=caption, label=f"tab:{name}"))
    print(f"\n=== {caption} ===\n{df.to_string(index=False)}\n -> {OUT}/{name}.csv/.tex", flush=True)


# ----------------------------------------------------------------------------- experiments
def exp_paramatch(p, a):
    """A: LSTM widened to BiLSTM size, and BiLSTM shrunk to LSTM size."""
    task = price_task(p)
    big = count_params("bilstm", 64, 32)        # 76,065
    small = count_params("lstm", 64, 32)        # 29,857
    u1L, u2L, cL = match_units("lstm", big)
    u1B, u2B, cB = match_units("bilstm", small)
    print(f"LSTM matched to BiLSTM ({big}): units {u1L}->{u2L}, {cL} params")
    print(f"BiLSTM matched to LSTM ({small}): units {u1B}->{u2B}, {cB} params")
    variants = {
        "LSTM (64-32)": ("lstm", DEFAULT),
        f"LSTM-wide ({u1L}-{u2L})": ("lstm", {**DEFAULT, "u1": u1L, "u2": u2L}),
        "BiLSTM (64-32)": ("bilstm", DEFAULT),
        f"BiLSTM-small ({u1B}-{u2B})": ("bilstm", {**DEFAULT, "u1": u1B, "u2": u2B}),
    }
    fc, ps, pr, ite = {}, {}, {}, None
    for name, (kind, cfg) in variants.items():
        print(f"  {name}")
        r = run_nn(task, kind, cfg, seeds=a.seeds, epochs=a.epochs)
        fc[name], ps[name], pr[name], ite = r["test"], r["per_seed"], r["params"], r["ite"]
    y = p[ite]
    fc = {"Naive": p[ite - 1], **fc}
    save(metrics_table(y, fc, ps, pr), "A_paramatch_metrics",
         "Parameter-matched comparison of LSTM and BiLSTM (test set)")
    names = list(variants)
    pairs = [(names[1], names[2]), (names[0], names[3]), (names[2], names[3]),
             (names[0], names[1]), ("Naive", names[1]), ("Naive", names[2])]
    save(dm_table(y, fc, pairs=pairs), "A_paramatch_dm", "DM tests, parameter-matched models")


def exp_hpsearch(p, a):
    """B: random search on the VALIDATION block only, then 5-seed test evaluation."""
    task = price_task(p)
    grid = dict(units=[(32, 16), (64, 32), (128, 64)], drop=[0.0, 0.1, 0.2, 0.3],
                lr=[1e-3, 5e-4, 1e-4], batch=[32, 64], T=[20, 40, 60, 90])
    rng = random.Random(42)
    configs = [DEFAULT]
    while len(configs) < a.n_configs:
        u = rng.choice(grid["units"])
        c = dict(u1=u[0], u2=u[1], drop=rng.choice(grid["drop"]), lr=rng.choice(grid["lr"]),
                 batch=rng.choice(grid["batch"]), T=rng.choice(grid["T"]))
        if c not in configs:
            configs.append(c)
    log, final, ps, pr, ite = [], {}, {}, {}, None
    for kind in ["lstm", "gru", "bilstm"]:
        best = None
        for i, cfg in enumerate(configs):
            print(f"  [{kind}] config {i+1}/{len(configs)}: {cfg}", flush=True)
            r = run_nn(task, kind, cfg, seeds=a.search_seeds, epochs=a.epochs, verbose=False)
            v = rmse(p[r["iva"]], r["val"])          # selection criterion: validation RMSE
            log.append(dict(model=kind, **cfg, val_RMSE=v))
            print(f"      val RMSE = {v:.4f}", flush=True)
            if best is None or v < best[0]:
                best = (v, cfg)
        print(f"  [{kind}] best on validation: {best[1]} (val RMSE {best[0]:.4f}); final 5-seed run")
        r = run_nn(task, kind, best[1], seeds=a.seeds, epochs=a.epochs)
        name = f"{kind.upper().replace('BILSTM', 'BiLSTM')}-tuned"
        final[name], ps[name], pr[name], ite = r["test"], r["per_seed"], r["params"], r["ite"]
        log.append(dict(model=kind + " (SELECTED)", **best[1], val_RMSE=best[0]))
    save(pd.DataFrame(log), "B_hpsearch_log", "Hyperparameter search log (validation RMSE, USD)")
    y = p[ite]
    xg, _ = run_xgb(task)
    fc = {"Naive": p[ite - 1], "XGBoost": xg, **final}
    save(metrics_table(y, fc, ps, pr), "B_tuned_metrics", "Tuned recurrent models (test set)")
    save(dm_table(y, fc), "B_tuned_dm", "DM tests, tuned recurrent models")


def exp_returns(p, a):
    """C: predict next-day log return; convert to price p_{t-1}*exp(r_hat)."""
    task = return_task(p)
    fc, ps = {}, {}
    xg, ite = run_xgb(task)
    for kind in ["lstm", "gru", "bilstm"]:
        print(f"  returns: {kind}")
        r = run_nn(task, kind, DEFAULT, seeds=a.seeds, epochs=a.epochs)
        name = kind.upper().replace("BILSTM", "BiLSTM") + "-ret"
        fc[name], ps[name], ite = r["test"], r["per_seed"], r["ite"]
    y = p[ite]
    fc = {"Naive": p[ite - 1], "XGBoost-ret": xg, **fc}
    tab = metrics_table(y, fc, ps)
    # directional accuracy on days with a non-zero move (naive has no direction -> skipped)
    true_dir = np.sign(y - p[ite - 1])
    da, pda = [], []
    for name, f in fc.items():
        pred_dir = np.sign(f - p[ite - 1])
        mask = (true_dir != 0) & (pred_dir != 0)
        if mask.sum() == 0:
            da.append(np.nan); pda.append(np.nan); continue
        hits = int((pred_dir[mask] == true_dir[mask]).sum())
        da.append(hits / mask.sum())
        pda.append(stats.binomtest(hits, int(mask.sum()), 0.5).pvalue)
    tab["DirAcc"], tab["p_DirAcc_vs_50%"] = da, pda
    save(tab, "C_returns_metrics", "Return-based target: price-level accuracy and directional accuracy")
    save(dm_table(y, fc, pairs=[("Naive", k) for k in fc if k != "Naive"]),
         "C_returns_dm", "DM tests vs naive, return-based models")


def exp_multistep(p, a):
    """D: direct h-step-ahead price forecasts."""
    rows, dms = [], []
    for h in a.horizons:
        task = price_task(p, h=h)
        xg, ite = run_xgb(task)
        fc = {"Naive": p[ite - h], "XGBoost": xg}
        for kind in a.ms_models:
            print(f"  h={h}: {kind}")
            r = run_nn(task, kind, DEFAULT, seeds=a.seeds, epochs=a.epochs)
            fc[kind.upper().replace("BILSTM", "BiLSTM")] = r["test"]
        y = p[ite]
        t = metrics_table(y, fc); t.insert(0, "h", h); rows.append(t)
        d = dm_table(y, fc, h=h, pairs=[("Naive", k) for k in fc if k != "Naive"])
        d.insert(0, "h", h); dms.append(d)
    save(pd.concat(rows), "D_multistep_metrics", "Direct multi-step forecasts (test set)")
    save(pd.concat(dms), "D_multistep_dm", "DM tests vs naive, multi-step (HLN, h-1 lags)")


def exp_exog(p, dates, a):
    """E: add one exogenous series (aligned to WTI dates, forward-filled = past values only)."""
    if a.synthetic:
        ex = pd.Series(np.cumsum(np.random.default_rng(1).normal(0, 0.3, len(p))) + 95, index=dates)
    else:
        ex = load_series(a.exog, a.exog_csv, start="1999-01-01")
    ex = ex.reindex(ex.index.union(dates)).sort_index().ffill().reindex(dates)
    ok = ~ex.isna().values
    if not ok.all():
        print(f"  dropping {(~ok).sum()} leading days without {a.exog} data")
    p2, e2 = p[ok], ex.values[ok]
    uni, multi = price_task(p2), price_task(p2, exog=e2)
    fc, ps = {}, {}
    xu, ite = run_xgb(uni)
    xm, _ = run_xgb(multi)
    fc["XGBoost"], fc[f"XGBoost+{a.exog}"] = xu, xm
    for kind in ["lstm", "gru", "bilstm"]:
        K = kind.upper().replace("BILSTM", "BiLSTM")
        for lab, task in [(K, uni), (f"{K}+{a.exog}", multi)]:
            print(f"  exog: {lab}")
            r = run_nn(task, kind, DEFAULT, seeds=a.seeds, epochs=a.epochs)
            fc[lab], ps[lab] = r["test"], r["per_seed"]
    y = p2[ite]
    fc = {"Naive": p2[ite - 1], **fc}
    save(metrics_table(y, fc, ps), "E_exog_metrics", f"Univariate vs. +{a.exog} (test set)")
    pairs = [(k, f"{k}+{a.exog}") for k in ["XGBoost", "LSTM", "GRU", "BiLSTM"]] + \
            [("Naive", f"{k}+{a.exog}") for k in ["XGBoost", "LSTM", "GRU", "BiLSTM"]]
    save(dm_table(y, fc, pairs=pairs), "E_exog_dm", f"DM tests, effect of adding {a.exog}")


# ----------------------------------------------------------------------------- main
def main(argv=None):
    import sys
    if argv is None and "ipykernel" in sys.modules:   # running inside a notebook cell
        argv = NOTEBOOK_ARGS
        print("Notebook mode, using NOTEBOOK_ARGS:", argv)
    ap = argparse.ArgumentParser()
    ap.add_argument("--exp", nargs="+", default=["all"],
                    choices=["all", "paramatch", "hpsearch", "returns", "multistep", "exog"])
    ap.add_argument("--csv", help="CSV with Date, Close for WTI (else Yahoo CL=F)")
    ap.add_argument("--exog", default="DX-Y.NYB", help="exogenous ticker (e.g. DX-Y.NYB, BZ=F, ^GSPC)")
    ap.add_argument("--exog_csv", help="CSV with Date, Close for the exogenous series")
    ap.add_argument("--seeds", type=int, nargs="+", default=SEEDS)
    ap.add_argument("--search_seeds", type=int, nargs="+", default=[0, 1])
    ap.add_argument("--n_configs", type=int, default=12)
    ap.add_argument("--horizons", type=int, nargs="+", default=[5, 10, 20])
    ap.add_argument("--ms_models", nargs="+", default=["lstm", "gru", "bilstm"])
    ap.add_argument("--epochs", type=int, default=100)
    ap.add_argument("--synthetic", action="store_true", help="smoke test on a random walk")
    a, unknown = ap.parse_known_args(argv)
    if unknown:
        print("Ignoring unknown arguments:", unknown)

    s = synthetic() if a.synthetic else load_series("CL=F", a.csv)
    p, dates = s.values.astype("float64"), s.index
    n_tr, n_va, n_te = split_sizes(len(p))
    print(f"{len(p)} days {dates[0].date()} .. {dates[-1].date()} | train {n_tr} val {n_va} test {n_te}")
    if not a.synthetic and len(p) != 6435:
        print("NOTE: series length differs from the paper (6,435). Pass --csv with the paper's data "
              "to reproduce exactly.")
    os.makedirs(OUT, exist_ok=True)
    json.dump(vars(a), open(f"{OUT}/run_args.json", "w"), indent=1)

    run = (lambda k: "all" in a.exp or k in a.exp)
    if run("paramatch"): exp_paramatch(p, a)
    if run("hpsearch"):  exp_hpsearch(p, a)
    if run("returns"):   exp_returns(p, a)
    if run("multistep"): exp_multistep(p, a)
    if run("exog"):      exp_exog(p, dates, a)

    import shutil
    zipped = shutil.make_archive("results", "zip", OUT)
    print(f"\nAll done. Download {os.path.abspath(zipped)} (Colab file panel) and send it back.")


if __name__ == "__main__":
    main()

Notebook mode, using NOTEBOOK_ARGS: ['--exp', 'paramatch', 'returns', 'hpsearch', '--csv', '/crude_oil_data (1).csv']
Loaded /crude_oil_data (1).csv: date column 'Price', price column 'Close', 6435 rows
6435 days 2000-08-23 .. 2026-04-10 | train 4504 val 965 test 966
LSTM matched to BiLSTM (76065): units 102->51, 74697 params
BiLSTM matched to LSTM (29857): units 40->20, 30273 params
  LSTM (64-32)
    lstm    seed 0: epochs= 43 test RMSE=2.2659 (74s)
    lstm    seed 1: epochs= 36 test RMSE=2.8480 (62s)
    lstm    seed 2: epochs= 30 test RMSE=3.9248 (54s)
    lstm    seed 3: epochs= 14 test RMSE=4.1151 (26s)
    lstm    seed 4: epochs= 36 test RMSE=2.9555 (60s)
  LSTM-wide (102-51)
    lstm    seed 0: epochs= 73 test RMSE=2.0714 (125s)
    lstm    seed 1: epochs= 44 test RMSE=2.3019 (76s)
    lstm    seed 2: epochs= 27 test RMSE=2.8700 (48s)
    lstm    seed 3: epochs= 41 test RMSE=2.6349 (68s)
    lstm    seed 4: epochs= 52 test RMSE=2.0695 (88s)
  BiLSTM (64-32)
    bilstm  seed 0: